In [1]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
import joblib
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.calibration import CalibratedClassifierCV
import seaborn as sns
import matplotlib.pyplot as plt


In [2]:

df = pd.read_csv('/Users/sambit_03/Desktop/Student Response/ASAG_math.csv')

df.rename(columns={
    "question":         "question",
    "reference_answer": "ref_ans",
    "provided_answer":   "response",
    "grade":            "grade",
    "data_source":   "subject"
}, inplace=True)

print(df.columns)

Index(['question', 'response', 'ref_ans', 'grade', 'subject',
       'normalized_grade', 'weight'],
      dtype='object')


In [3]:
print(df["subject"].value_counts())

subject
SciEntsBank    9684
Beetle         3567
SAF            2237
DigiKlausur     584
Mohler          575
Stita           293
CU-NLP          149
Name: count, dtype: int64


In [4]:
df_math = df[df["subject"].str.contains("math|beetle|sciEnts", case=False, na=False)]
print(f"Math/Science subset shape: {df_math.shape}")


Math/Science subset shape: (13251, 7)


In [5]:
def map_grade(score):
    if score >= 0.9:
        return "correct"
    elif score >= 0.5:
        return "partially_correct_incomplete"
    elif score < 0.2:  # Adjusted down to 0.2 to safely capture very low scores/irrelevant answers
        return "irrelevant"
    else:
        return "contradictory" # This catches everything else between 0.2 and 0.5

In [6]:
df["grade_label"] = df["grade"].apply(map_grade)
print("\nGrade label distribution:")
print(df["grade_label"].value_counts())


Grade label distribution:
grade_label
correct                         12920
irrelevant                       3042
partially_correct_incomplete      980
contradictory                     147
Name: count, dtype: int64


In [7]:
df[["question", "ref_ans", "response", "grade_label", "subject"]].to_csv(
    "/Users/sambit_03/Desktop/Student Response/preprocessed.csv",
    index=False
)

In [8]:
data = pd.read_csv('/Users/sambit_03/Desktop/Student Response/preprocessed.csv')
print(data["grade_label"].value_counts())
print(data.shape)
model = SentenceTransformer('all-MiniLM-L6-v2')

ref_encoding  = model.encode(data['ref_ans'].astype(str).tolist(),  
                             convert_to_numpy=True, show_progress_bar=True)
resp_encoding = model.encode(data['response'].astype(str).tolist(), 
                             convert_to_numpy=True, show_progress_bar=True)

grade_label
correct                         12920
irrelevant                       3042
partially_correct_incomplete      980
contradictory                     147
Name: count, dtype: int64
(17089, 5)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/535 [00:00<?, ?it/s]

Batches:   0%|          | 0/535 [00:00<?, ?it/s]

In [9]:
cos_sim = np.sum(ref_encoding * resp_encoding, axis=1) / (
    np.linalg.norm(ref_encoding,  axis=1) *
    np.linalg.norm(resp_encoding, axis=1)
)
print("Cosine sim range:", cos_sim.min().round(3), "→", cos_sim.max().round(3))

Cosine sim range: -0.106 → 1.0


In [10]:
n = len(data)
combined        = np.vstack([ref_encoding, resp_encoding])
scaler          = StandardScaler()
combined_scaled = scaler.fit_transform(combined)
pca             = PCA(n_components=50, random_state=42)
combined_pca    = pca.fit_transform(combined_scaled)
print(f"384 dims → {combined_pca.shape[1]} dims | variance kept: {pca.explained_variance_ratio_.sum():.4f}")
ref_pca  = combined_pca[:n]
resp_pca = combined_pca[n:]

384 dims → 50 dims | variance kept: 0.6466


In [11]:
X = np.hstack([ref_pca, resp_pca, cos_sim.reshape(-1, 1)])

ref_cols      = [f"ref_pc_{i}"  for i in range(ref_pca.shape[1])]
resp_cols     = [f"resp_pc_{i}" for i in range(resp_pca.shape[1])]
feature_names = ref_cols + resp_cols + ["cosine_similarity"]

X_df = pd.DataFrame(X, columns=feature_names)
print(f"Final feature matrix: {X_df.shape}")
le = LabelEncoder()
y  = le.fit_transform(data['grade_label'])
print("Classes:", le.classes_) 

Final feature matrix: (17089, 101)
Classes: ['contradictory' 'correct' 'irrelevant' 'partially_correct_incomplete']


In [12]:
y_df = pd.DataFrame({
    "grade":         data["grade_label"],
    "grade_encoded": y,
    "subject":       data["subject"]   # keep subject for threshold calibration
})

In [13]:
BASE = "/Users/sambit_03/Desktop/Student Response/"

X_df.to_csv(BASE + "X_features.csv", index=False)
y_df.to_csv(BASE + "y_labels.csv",   index=False)

joblib.dump(scaler, BASE + "scaler.pkl")
joblib.dump(pca,    BASE + "pca.pkl")
joblib.dump(le,     BASE + "label_encoder.pkl")

print("All saved.")

All saved.


In [14]:
BASE = "/Users/sambit_03/Desktop/Student Response/"

In [15]:
X       = pd.read_csv(BASE + "X_features.csv")
y_df    = pd.read_csv(BASE + "y_labels.csv")
y       = y_df["grade_encoded"]
subject = y_df["subject"]

In [16]:
X_train, X_test, y_train, y_test, subj_train, subj_test = train_test_split(
    X, y, subject,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train = X_train.to_numpy(dtype=np.float32)   
X_test  = X_test.to_numpy(dtype=np.float32)

y_train = y_train.to_numpy().astype(np.int32)
y_test  = y_test.to_numpy().astype(np.int32)


In [ ]:
print(X_train.shape, X_test.shape)

(13671, 101) (3418, 101)


: 

In [ ]:
model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softprob',
    eval_metric='mlogloss',
    random_state=42,
    tree_method='hist',   # memory-efficient histogram method
    device='cpu',         # force CPU, avoids Metal/MPS conflicts on M2
    n_jobs=1              # no parallelism — prevents memory spike
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)
